In [1]:
import math
from pprint import pprint

import keyboard
import meshcat_shapes
import numpy as np
import pink
import pinocchio as pin
import plotly.graph_objects as go
import qpsolvers
from loop_rate_limiters import RateLimiter
from pink import solve_ik
from pink.tasks import FrameTask, PostureTask
from pink.utils import custom_configuration_vector
from pink.visualization import start_meshcat_visualizer
from robot_descriptions.loaders.pinocchio import load_robot_description
import numpy.typing as npt
from scipy.spatial.transform import Rotation as R, Slerp
import tkinter as tk
from tkinter import messagebox

from rtde_control import RTDEControlInterface
from rtde_receive import RTDEReceiveInterface

In [2]:
def ask_if_restart_viz():
    # 1. Create a hidden Tkinter root window
    root = tk.Tk()
    root.withdraw()

    # 2. Force the root window to the front
    root.attributes("-topmost", True)

    # 3. Display the confirm box (similar to pyautogui.confirm)
    response = messagebox.askokcancel("Confirmation", "Do you want to proceed?")

    # 4. Clean up
    root.destroy()

    if response:
        print("User clicked OK")
    else:
        print("User clicked Cancel")
        
    return response

In [ ]:
robot_joint_pos = {
    "top": (
        math.radians(-90.17),
        math.radians(-105.66),
        math.radians(-63),
        math.radians(-101.35),
        math.radians(90),
        math.radians(0),
    ),
}

joint_names = [
    "shoulder_pan_joint",
    "shoulder_lift_joint",
    "elbow_joint",
    "wrist_1_joint",
    "wrist_2_joint",
    "wrist_3_joint",
]

# robot = load_robot_description("ur5_official_description")
# robot.model:
# Nb joints = 7 (nq=6,nv=6)
#   Joint 0 universe: parent=0
#   Joint 1 shoulder_pan_joint: parent=0
#   Joint 2 shoulder_lift_joint: parent=1
#   Joint 3 elbow_joint: parent=2
#   Joint 4 wrist_1_joint: parent=3
#   Joint 5 wrist_2_joint: parent=4
#   Joint 6 wrist_3_joint: parent=5

robot = load_robot_description("ur5e_description")

try:
    viewer_configs
except NameError:
    viewer_configs: dict[str, dict] = {}

In [4]:
def set_target_0(configuration, end_effector_task):
    X_W_B = configuration.get_transform_frame_to_world("base")

    # X_base_to_target = X_B_W * X_W_target
    target_wrt_base = X_W_B.inverse() * end_effector_task.transform_target_to_world

    # take and keep the current orient. and trans. of Tool relative to base
    # set that tool is at z-=-0.5 in base frame
    target_wrt_base.translation[2] = -0.5
        
    # the api requires relative to world
    target_wrt_world = X_W_B * target_wrt_base

    end_effector_task.set_target(target_wrt_world)
    
    return target_wrt_world

def set_target_1(configuration, end_effector_task):
    T_wt = configuration.get_transform_frame_to_world("tool0")
    T_wt.translation[2] -= 0.5
    end_effector_task.set_target(T_wt)
    return T_wt

## move -50 cm below base in z

In [47]:
end_effector_task = FrameTask(
    "tool0",
    position_cost=1.0,  # [cost] / [m]
    orientation_cost=1.0,  # [cost] / [rad]
    lm_damping=1.0,  # tuned for this setup
)

posture_task = PostureTask(
    cost=1e-3,  # [cost] / [rad]
)

# end_effector_task = FrameTask(
#     "tool0",
#     position_cost=10.0,     # higher position tracking priority
#     orientation_cost=2.0,   # orientation tracking
#     lm_damping=1e-2,
# )
# posture_task = PostureTask(cost=1e-4)

tasks = [end_effector_task, posture_task]

# set initial configuration for visualization
q_kwargs = dict(zip(joint_names, robot_joint_pos["top"]))
q_ref = custom_configuration_vector(robot, **q_kwargs)

configuration = pink.Configuration(robot.model, robot.data, q_ref)
for task in tasks:
    task.set_target_from_configuration(configuration)

# Select QP solver
solver = (
    "daqp" if "daqp" in qpsolvers.available_solvers else qpsolvers.available_solvers[0]
)


rate = RateLimiter(frequency=200.0, warn=False)
dt = rate.period

pos_tol = 1e-3  # 1 mm
rot_tol = 1e-2  # ~0.57 deg
stable_steps = 20  # avoid stopping on one noisy step

# pyautogui message if want to restart visualizer
if ask_if_restart_viz():
    viz = start_meshcat_visualizer(robot)
    

viz.display(configuration.q)  # q is the same a s q_ref
viewer = viz.viewer

for x in viewer_configs.keys():
    viewer[x].delete()

User clicked OK
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7001/static/


In [36]:
ok_count = 0
t = 0.0  # [s]
for x in viewer_configs.keys():
    viewer[x].delete()
    
target_wrt_world = set_target_0(configuration, end_effector_task)
viewer_configs = {
    "end_effector_target": {"opacity": 0.5},
    "end_effector": {"opacity": 1.0},
}
for view_name, view_config in viewer_configs.items():
    meshcat_shapes.frame(viewer[view_name], **view_config)

while True:
    if keyboard.is_pressed("Esc"):
        print("Exiting loop.")
        break
    
    # solve + integrate
    velocity = solve_ik(configuration, tasks, dt, solver=solver)
    configuration.integrate_inplace(velocity, dt)
    
    # Check convergence (world frame)
    current_pose = configuration.get_transform_frame_to_world("tool0")
    target_pose = end_effector_task.transform_target_to_world

    pos_err = np.linalg.norm(target_pose.translation - current_pose.translation)
    rot_err = np.linalg.norm(pin.log3(target_pose.rotation.T @ current_pose.rotation))
    
    if pos_err < pos_tol and rot_err < rot_tol:
        ok_count += 1
    else:
        ok_count = 0
        
    if ok_count >= stable_steps:
        print(f"Target reached. pos_err={pos_err:.6f} m, rot_err={rot_err:.6f} rad")
        break
    
    # Visualization (Meshcat always expects world-frame transforms)
    viewer["end_effector_target"].set_transform(target_pose.np)
    viewer["end_effector"].set_transform(current_pose.np)
    viz.display(configuration.q)
    rate.sleep()
    t += dt

Target reached. pos_err=0.000005 m, rot_err=0.000002 rad


## linear move sim without orientation change

In [43]:
ok_count = 0
t = 0.0  # [s]
v_cart = 0.5   # try 0.01 ~ 0.05 first Cartesian line speed [m/s]

start_pose = configuration.get_transform_frame_to_world("tool0")
target_wrt_world = set_target_0(configuration, end_effector_task)
goal_pose = target_wrt_world  # already computed in your code

p0 = start_pose.translation.copy()
p1 = goal_pose.translation.copy()

R0 = start_pose.rotation.copy()
R1 = goal_pose.rotation.copy()   # same as R0 in your current setup

delta = p1 - p0
distance = np.linalg.norm(delta)

for x in viewer_configs.keys():
    viewer[x].delete()

viewer_configs = {
    "end_effector_target": {"opacity": 0.5},
    "end_effector": {"opacity": 1.0},
    "goal": {"opacity": 1.0},
}
for view_name, view_config in viewer_configs.items():
    meshcat_shapes.frame(viewer[view_name], **view_config)

if distance < 1e-9:
    print("Already at target.")
else:
    direction = delta / distance
    s = 0.0         # traveled distance along line

    while True:
        if keyboard.is_pressed("Esc"):
            print("Exiting loop.")
            break

        # Advance along the straight line
        s = min(s + v_cart * dt, distance)
        p_des = p0 + direction * s

        # Keep orientation fixed for now
        target_step = pin.SE3(R0, p_des)

        # Update IK target every cycle
        end_effector_task.set_target(target_step)

        # Solve IK and integrate
        velocity = solve_ik(configuration, tasks, dt, solver=solver)
        configuration.integrate_inplace(velocity, dt)

        # Check tracking error
        current_pose = configuration.get_transform_frame_to_world("tool0")
        pos_err = np.linalg.norm(target_step.translation - current_pose.translation)
        rot_err = np.linalg.norm(pin.log3(target_step.rotation.T @ current_pose.rotation))

        if s >= distance and pos_err < pos_tol and rot_err < rot_tol:
            ok_count += 1
        else:
            ok_count = 0

        # Visualization
        viewer["end_effector_target"].set_transform(target_step.np)
        viewer["end_effector"].set_transform(current_pose.np)
        viewer["goal"].set_transform(goal_pose.np)
        viz.display(configuration.q)

        if ok_count >= stable_steps:
            print(f"Line target reached. pos_err={pos_err:.6f} m, rot_err={rot_err:.6f} rad")
            break
        
        rate.sleep()
        t += dt

Line target reached. pos_err=0.000005 m, rot_err=0.000003 rad


## linear with orientation change

In [46]:
# Longer duration = easier tracking, straighter path.
# Example: 5 cm/s nominal translational speed
v_cart = 0.5  # m/s
min_duration = 2.0

# ============================================================
# 5) Start pose
# ============================================================
# T_wt
start_pose = configuration.get_transform_frame_to_world("tool0")
p0 = start_pose.translation.copy()
R0 = start_pose.rotation.copy()

goal_translation = p0 + np.array([0.0, 0.0, -0.80])

R_delta = R.from_euler("z", 90, degrees=True).as_matrix()
goal_rotation = R_delta @ R0

goal_pose = pin.SE3(goal_rotation, goal_translation)

print("Goal pose:\n", goal_pose)

# ============================================================
# 7) Build SLERP for orientation interpolation
# ============================================================
key_times = [0.0, 1.0]
key_rots = R.from_matrix(np.stack([R0, goal_rotation], axis=0))
slerp = Slerp(key_times, key_rots)

# ============================================================
# 8) Trajectory timing
#    alpha in [0, 1]
# ============================================================
distance = np.linalg.norm(goal_translation - p0)
if distance < 1e-9:
    total_duration = min_duration
else:
    total_duration = max(distance / v_cart, min_duration)

print(f"Distance = {distance:.4f} m")
print(f"Total duration = {total_duration:.3f} s")


# ============================================================
# 10) Main loop
# ============================================================
ok_count = 0
t = 0.0


viewer_configs = {
    "end_effector_target": {"opacity": 0.5},
    "end_effector": {"opacity": 1.0},
    "tcp_target_step": {"opacity": 0.3},
}
for view_name, view_config in viewer_configs.items():
    meshcat_shapes.frame(viewer[view_name], **view_config)
    
while True:
    if keyboard.is_pressed("Esc"):
        print("Exiting loop.")
        break

    # Progress parameter alpha from 0 -> 1
    alpha = min(t / total_duration, 1.0)

    # ---- Translation: linear interpolation ----
    p_des = (1.0 - alpha) * p0 + alpha * goal_translation

    # ---- Orientation: spherical linear interpolation (SLERP) ----
    R_des = slerp([alpha]).as_matrix()[0]

    # Desired pose at this time step
    target_step = pin.SE3(R_des, p_des)

    # Update end-effector target every cycle
    end_effector_task.set_target(target_step)

    # Solve IK and integrate
    velocity = solve_ik(configuration, tasks, dt, solver=solver)
    configuration.integrate_inplace(velocity, dt)

    # Current pose
    current_pose = configuration.get_transform_frame_to_world("tool0")

    # Tracking errors (relative to current interpolated target)
    pos_err = np.linalg.norm(target_step.translation - current_pose.translation)
    rot_err = np.linalg.norm(pin.log3(target_step.rotation.T @ current_pose.rotation))

    # Stop only when final alpha reached and robot is stably close to final goal
    final_pos_err = np.linalg.norm(goal_pose.translation - current_pose.translation)
    final_rot_err = np.linalg.norm(
        pin.log3(goal_pose.rotation.T @ current_pose.rotation)
    )

    if alpha >= 1.0 and final_pos_err < pos_tol and final_rot_err < rot_tol:
        ok_count += 1
    else:
        ok_count = 0

    if ok_count >= stable_steps:
        print(
            f"Goal reached.\n"
            f"  final_pos_err = {final_pos_err:.6f} m\n"
            f"  final_rot_err = {final_rot_err:.6f} rad"
        )
        break

    # Visualization
    viewer["end_effector_target"].set_transform(goal_pose.np)
    viewer["tcp_target_step"].set_transform(target_step.np)
    # viewer["tcp_goal"].set_transform(goal_pose.np)
    viewer["end_effector"].set_transform(current_pose.np)
    viz.display(configuration.q)

    rate.sleep()
    t += dt


Goal pose:
   R =
  0.00296706     0.999996  0.000174532
    0.999996  -0.00296706 -5.18055e-07
-2.05139e-10  0.000174533           -1
  p =  0.135077  0.598547 -0.250775

Distance = 0.8000 m
Total duration = 2.000 s
Goal reached.
  final_pos_err = 0.000006 m
  final_rot_err = 0.000003 rad
